In [1]:
import pandas as pd
import numpy as np
import sys
from numba import njit, prange
parent_path = '/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline'
sys.path.append(parent_path)
from SynOmics.synthesizer.Synthpopsynthesizer import SynthpopSynthesizer
from SynOmics.processing.metadata import MetaData

metadata_path = "OriginalData/feature_metadata.json"
data_path = "OriginalData/integrated_data_final.csv"

or_df = pd.read_csv(data_path, index_col = 0)
metadata = MetaData.load(metadata_path)
grouped_metadata = MetaData.grouping_features_astype(or_df, metadata)
categorical_features = grouped_metadata.get("ordinal_categorical") + grouped_metadata.get("dummy_categorical") + grouped_metadata.get("missing_categorical")
numerical_features = grouped_metadata.get("numerical")

In [2]:
predictor_matrix = np.load("synthpop/predictor_matrix.npy")
sorted_features_indices = []
with open("synthpop/sorted_features.txt", "r") as f:
    for line in f:
        sorted_features_indices.append(int(line.strip())) 
sorted_features_names = [or_df.columns[int(i)] for i in sorted_features_indices]

predictor_df_from_dict = pd.DataFrame(predictor_matrix, 
                                      index=sorted_features_names, 
                                      columns=sorted_features_names)

In [11]:
ordered_df = or_df[sorted_features_names].iloc[:,0:1000]
pred_df = predictor_df_from_dict.iloc[0:1000,0:1000]
grouped_metadata = MetaData.grouping_features_astype(ordered_df, metadata)
categorical_features = grouped_metadata.get("ordinal_categorical") + grouped_metadata.get("dummy_categorical") + grouped_metadata.get("missing_categorical")
numerical_features = grouped_metadata.get("numerical")

In [5]:
from SynOmics.synthesizer.Synthpopsynthesizer import SynthpopSynthesizer
ordered_df = or_df[sorted_features_names]

In [14]:
synth = SynthpopSynthesizer(
    output_path = "synthpop",
    metadata = metadata,
    r_home = "/opt/R/4.4.1/lib/R",
    r_terminal = "R441"
)
synthetic_data = synth.generate(
    data = ordered_df, 
    data_ids = ordered_df.index.tolist(),
    enforce_rounding = False, 
    enforce_min_max = False, 
    masking = False, 
    seed = 42, 
    n_samples = ordered_df.shape[0], 
    fit_params = None,
    sample_params = {
        "discrete_columns": categorical_features,
        "method": "cart",
        "minimumlevels": 3,
        "proper": False,
        "n_datasets": 1,
        "verbose": True,
        "predictor_matrix": pred_df
    }, 
    output_filename = 'synthpop_synthetic_data.csv', 
    save_index = False
)

2025-10-21 14:31:42 - DEBUG - Standalone logger initialized successfully.
2025-10-21 14:31:42 - INFO - ========== Synthesizer Initialized ==========
2025-10-21 14:31:42 - INFO - Class: SynthpopSynthesizer
2025-10-21 14:31:42 - INFO - Output path: synthpop
2025-10-21 14:31:42 - INFO - Log file: synthpop/SynthpopSynthesizer_140230137918272.log
2025-10-21 14:31:42 - INFO - Metadata provided with 40992 columns.
2025-10-21 14:31:42 - INFO - ============================================
2025-10-21 14:31:42 - INFO - R_HOME set to: /opt/R/4.4.1/lib/R
--- System & Process Info ---
Current Date and Time (UTC): 2025-10-21 12:31:42
Current User's Login: trinhtc
CPU Model: x86_64
Physical Cores: 32
Logical Processors: 32
Process RAM before execution: 13507.02 MB

--- GPU Info ---
GPU monitoring failed. Ensure 'nvidia-ml-py' is installed and NVIDIA drivers are accessible.
Error: module 'nvidia_smi' has no attribute 'nvidia_smi_lib'

--- Function Execution ---
2025-10-21 14:31:42 - INFO - Pipeline sta